# 04a · Recommendation with Spark MLlib ALS
Implicit-feedback ALS on the user–product matrix from notebook 02 (weights 1/3/5 for view/cart/purchase, train = before 2019-11-24, test = purchases in the last 7 days). Evaluated on top-10 recommendations with already-purchased products removed.

Also computes a **popularity baseline** (most-purchased products in train). `04b` runs the same protocol with the `implicit` library, without Spark. Metrics go to `ml/outputs/04a_metrics.json` (written after every fraction, so a crash on the largest run keeps earlier results).

In [1]:
import os, json, time
import numpy as np, pandas as pd
from IPython.display import display
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.ml.recommendation import ALS
from pyspark import StorageLevel

DATA, OUT, FIG = "/home/jovyan/work/data", "/home/jovyan/work/ml/outputs", "/home/jovyan/work/laporan/gambar"
GOLD = f"{DATA}/gold"
SEED, RANK, MAXITER, ALPHA, REG, TOPN = 42, 32, 8, 10.0, 0.05, 10
# a user belongs to fraction p (percent) when pmod(hash(user_id), 100) < p; subsets are nested
FRACTIONS = [2, 10, 25, 100]

spark = (SparkSession.builder.master("local[*]").appName("04a_als_mllib")
         .config("spark.driver.memory", "13g")
         .config("spark.sql.shuffle.partitions", "64")
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
train_all = spark.read.parquet(f"{GOLD}/als_train")
test_all = spark.read.parquet(f"{GOLD}/als_test")
metrics = dict(params=dict(rank=RANK, maxIter=MAXITER, alpha=ALPHA, regParam=REG, topN=TOPN), runs=[])

## Evaluation helpers
Per test user: Precision@10, Recall@10, NDCG@10 and HitRate@10 against the products bought in the test week. Averages are over test users in the subset.

In [2]:
def subset(df, p):
    return df if p == 100 else df.filter(F.pmod(F.xxhash64("user_id"), F.lit(100)) < p)

def evaluate(recs, test):
    # recs: user_id, product_id, rank (1..TOPN); test: user_id, product_id
    hits = (recs.join(test.select("user_id", "product_id").withColumn("rel", F.lit(1)), ["user_id", "product_id"], "left")
                .withColumn("gain", F.when(F.col("rel") == 1, 1.0 / F.log2(F.col("rank") + 1.0)).otherwise(0.0))
                .groupBy("user_id").agg(F.sum(F.coalesce("rel", F.lit(0))).alias("hits"), F.sum("gain").alias("dcg")))
    nrel = test.groupBy("user_id").agg(F.count("*").alias("n_rel"))
    idcg = F.udf(lambda n: float(sum(1.0 / np.log2(i + 2) for i in range(min(n, TOPN)))), "double")
    per_user = (nrel.join(hits, "user_id", "left").na.fill(0)
                .withColumn("p", F.col("hits") / TOPN).withColumn("r", F.col("hits") / F.col("n_rel"))
                .withColumn("ndcg", F.col("dcg") / idcg("n_rel")).withColumn("hit", (F.col("hits") > 0).cast("double")))
    row = per_user.agg(F.count("*").alias("users"), F.avg("p").alias("precision"), F.avg("r").alias("recall"),
                       F.avg("ndcg").alias("ndcg"), F.avg("hit").alias("hit_rate")).first().asDict()
    return {k: float(v) for k, v in row.items()}

def top_n_excluding_bought(recs100, train_buys):
    # recs100: user_id, recommendations[array<struct<product_id, rating>>]
    ex = recs100.select("user_id", F.posexplode("recommendations").alias("pos", "rec")) \
                .select("user_id", F.col("rec.product_id").alias("product_id"), F.col("pos"))
    ex = ex.join(train_buys, ["user_id", "product_id"], "left_anti")
    w = Window.partitionBy("user_id").orderBy("pos")
    return ex.withColumn("rank", F.row_number().over(w)).filter(F.col("rank") <= TOPN).select("user_id", "product_id", "rank")

## Popularity baseline
The same top-10 list (most purchased products in train, minus those the user already bought) for every test user. Any personalised model has to beat this.

In [3]:
pop_all = (train_all.filter("n_purchase > 0").groupBy("product_id").agg(F.sum("n_purchase").alias("buys"))
           .orderBy(F.desc("buys")).limit(TOPN + 100).toPandas())
pop_list = pop_all.product_id.tolist()
pop_df = spark.createDataFrame(pd.DataFrame({"pos": range(len(pop_list)), "product_id": pop_list}))

## ALS fits on nested user subsets (2%, 10%, 25%, 100%)
`recommendForUserSubset` scores each test user against all products; 100 candidates are requested so that already-bought items can be filtered and still leave 10.

In [4]:
def run(p):
    tr = subset(train_all, p).persist(StorageLevel.MEMORY_AND_DISK)
    te = subset(test_all, p).persist(StorageLevel.MEMORY_AND_DISK)
    n_tr, n_te_users = tr.count(), te.select("user_id").distinct().count()
    t0 = time.time()
    als = ALS(userCol="user_id", itemCol="product_id", ratingCol="weight", implicitPrefs=True,
              rank=RANK, maxIter=MAXITER, regParam=REG, alpha=ALPHA, seed=SEED,
              coldStartStrategy="drop", numUserBlocks=40, numItemBlocks=40)
    model = als.fit(tr); fit_s = time.time() - t0
    t0 = time.time()
    buys = tr.filter("n_purchase > 0").select("user_id", "product_id")
    users = te.select("user_id").distinct()
    recs = top_n_excluding_bought(model.recommendForUserSubset(users, TOPN + 100), buys).persist(StorageLevel.MEMORY_AND_DISK)
    m_als = evaluate(recs, te); rec_s = time.time() - t0
    # popularity baseline for the same users
    pop_recs = (users.crossJoin(pop_df.withColumnRenamed("pos", "pos_"))
                .join(buys, ["user_id", "product_id"], "left_anti"))
    w = Window.partitionBy("user_id").orderBy("pos_")
    pop_recs = pop_recs.withColumn("rank", F.row_number().over(w)).filter(F.col("rank") <= TOPN).select("user_id", "product_id", "rank")
    m_pop = evaluate(pop_recs, te)
    rec = dict(fraction_pct=p, train_pairs=n_tr, test_users=n_te_users, fit_s=round(fit_s, 1), recommend_eval_s=round(rec_s, 1),
               als=m_als, popularity=m_pop)
    recs.unpersist(); tr.unpersist(); te.unpersist()
    return rec

for p in FRACTIONS:
    try:
        r = run(p)
    except Exception as e:
        r = dict(fraction_pct=p, error=str(e)[:300])
    metrics["runs"].append(r); print(r)
    json.dump(metrics, open(f"{OUT}/04a_metrics.json", "w"), indent=2, default=str)
    if "error" in r: break

{'fraction_pct': 2, 'train_pairs': 1013419, 'test_users': 1623, 'fit_s': 7.7, 'recommend_eval_s': 6.4, 'als': {'users': 1623.0, 'precision': 0.019963031423290208, 'recall': 0.14759448378642226, 'ndcg': 0.09176217569778837, 'hit_rate': 0.18299445471349354}, 'popularity': {'users': 1623.0, 'precision': 0.016081330868761592, 'recall': 0.1081889903605961, 'ndcg': 0.06410587512462783, 'hit_rate': 0.1423290203327172}}
{'fraction_pct': 10, 'train_pairs': 5062861, 'test_users': 8026, 'fit_s': 16.6, 'recommend_eval_s': 13.2, 'als': {'users': 8026.0, 'precision': 0.02007226513830053, 'recall': 0.14784010776460718, 'ndcg': 0.09067951574862308, 'hit_rate': 0.18290555693994517}, 'popularity': {'users': 8026.0, 'precision': 0.01579865437328691, 'recall': 0.10817818937418142, 'ndcg': 0.06420058400791945, 'hit_rate': 0.13929728382756043}}
{'fraction_pct': 25, 'train_pairs': 12619596, 'test_users': 20084, 'fit_s': 39.0, 'recommend_eval_s': 33.1, 'als': {'users': 20084.0, 'precision': 0.0195379406492730

{'fraction_pct': 25, 'train_pairs': 12619596, 'test_users': 20084, 'fit_s': 32.0, 'recommend_eval_s': 28.3, 'als': {'users': 20084.0, 'precision': 0.019537940649273042, 'recall': 0.14708803204346754, 'ndcg': 0.09081662096452518, 'hit_rate': 0.1797450707030472}, 'popularity': {'users': 20084.0, 'precision': 0.015714001194981148, 'recall': 0.11093145222764365, 'ndcg': 0.06495648719758113, 'hit_rate': 0.1412069308902609}}


{'fraction_pct': 100, 'train_pairs': 50456607, 'test_users': 79571, 'fit_s': 149.8, 'recommend_eval_s': 108.1, 'als': {'users': 79571.0, 'precision': 0.01945558055070319, 'recall': 0.14646858363507037, 'ndcg': 0.08934918516829543, 'hit_rate': 0.17955033869123174}, 'popularity': {'users': 79571.0, 'precision': 0.015391285769941102, 'recall': 0.10976007916552977, 'ndcg': 0.06442329000411517, 'hit_rate': 0.14006359100677382}}


In [5]:
display(pd.json_normalize(metrics["runs"]).round(4).T)
spark.stop()

,0,1,2,3
fraction_pct,2.000000e+00,1.000000e+01,2.500000e+01,1.000000e+02
train_pairs,1.013419e+06,5.062861e+06,1.261960e+07,5.045661e+07
test_users,1.623000e+03,8.026000e+03,2.008400e+04,7.957100e+04
fit_s,7.700000e+00,1.660000e+01,3.900000e+01,1.868000e+02
recommend_eval_s,6.400000e+00,1.320000e+01,3.310000e+01,1.447000e+02
als.users,1.623000e+03,8.026000e+03,2.008400e+04,7.957100e+04
als.precision,2.000000e-02,2.010000e-02,1.950000e-02,1.950000e-02
als.recall,1.476000e-01,1.478000e-01,1.471000e-01,1.465000e-01
als.ndcg,9.180000e-02,9.070000e-02,9.080000e-02,8.930000e-02
als.hit_rate,1.830000e-01,1.829000e-01,1.797000e-01,1.796000e-01
